#  CNN Transfer Learning with VGG19 on CIFAR-100

This notebook demonstrates how to use CNNs and transfer learning to classify the CIFAR-100 dataset.

It does **not** focus on architecture innovation or optimization. Instead, it is a teaching-oriented practical example that can be quickly adapted to many development scenarios. The code and workflow can be reused to fine-tune mature industrial models for different tasks.

This notebook presents three transfer learning strategies:

1. **Method 1** — Replace the classifier head in a standard PyTorch VGG19 model.
2. **Method 2** — Keep only part of the VGG19 feature extractor, then rebuild the classifier and train a new head.
3. **Method 3** — Replace the classifier head and also unfreeze selected convolutional layers for manual fine-tuning.

The original project was built for .  
Reference article:

.........................................................

## 1. Download the dataset

If the  environment has limited download speed, it is recommended to download the dataset locally in advance and upload it manually into the `./data` folder. PyTorch and `torchvision` can then automatically recognize it.

You can also download the CIFAR-100 archive manually:

```bash
wget http://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vgg19, VGG19_Weights
import matplotlib.pyplot as plt
import numpy as np
import os

# ------------------------------
# 1. Download CIFAR-100 dataset
# ------------------------------
print("Step 1: Downloading CIFAR-100 dataset...")

# Transform for loading (only ToTensor for now, we'll resize later)
simple_transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                         download=True, transform=simple_transform)
testset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                        download=True, transform=simple_transform)

## 2. Randomly inspect a few samples

In [ ]:
# ------------------------------
# 2. Randomly select two examples and display them
# ------------------------------
print("\nStep 2: Randomly selecting two training examples...")

def imshow(img, title=None):
    """Display a tensor image (C, H, W)."""
    img = img.numpy().transpose((1, 2, 0))  # convert to H,W,C
    # CIFAR images are in [0,1], but may have slight clipping; clamp for display
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis('off')
    plt.show()

# Randomly pick two indices from training set
indices = np.random.choice(len(trainset), 2, replace=False)
for idx in indices:
    image, label = trainset[idx]
    label_name = trainset.classes[label]
    print(f"Index: {idx}, Label: {label} ({label_name})")
    imshow(image, title=f"Label: {label_name}")

## 3. Prepare the training data

VGG19 expects `224 × 224` RGB images normalized with ImageNet statistics.

In [ ]:
# ------------------------------
# 3. Prepare data for PyTorch (resize, normalize, batch)
# ------------------------------
print("\nStep 3: Preparing data loaders for VGG19...")

# VGG19 expects 224x224 RGB images, normalized with ImageNet stats
transform_train = transforms.Compose([
    transforms.Resize(256),          # Resize smaller side to 256
    transforms.RandomCrop(224),      # Random crop of 224x224
    transforms.RandomHorizontalFlip(),  # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Recreate datasets with proper transforms
trainset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                         download=False, transform=transform_train)
testset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                        download=False, transform=transform_test)

# Data loaders
batch_size = 64
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

print(f"Train samples: {len(trainset)}, Test samples: {len(testset)}")
print(f"Batch size: {batch_size}")

## 4. Load VGG19 and inspect its architecture

In [ ]:
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)

# Print entire architecture
print(model)

# # Access specific parts
# print(model.features)          # all conv/pool layers
# print(model.avgpool)           # AdaptiveAvgPool2d
# print(model.classifier)        # FC layers

# Access individual layers by index
first_conv = model.features[0]
last_conv = model.features[36]   # the final MaxPool2d
first_fc = model.classifier[0]

## 5. Three transfer learning methods

### Method 1 — Replace the classifier head

This is the classic approach: freeze the convolutional feature extractor and replace the final classifier layer for CIFAR-100.

In [ ]:
# Method 1: Replace the entire classifier

# ------------------------------
# 4. Write a CNN model (modified VGG19)
# ------------------------------
print("\nStep 4: Building VGG19 model for CIFAR-100...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained VGG19 (with batch norm? VGG19 has no BN by default, but we can use vgg19_bn if desired)
# We'll use standard VGG19 pretrained on ImageNet
weights = VGG19_Weights.IMAGENET1K_V1
model = vgg19(weights=weights)

# Freeze feature extractor (all convolutional layers)
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 100)  # CIFAR-100 has 100 classes

model = model.to(device)

# Print model summary (optional - uncomment if you have torchsummary)
# from torchsummary import summary
# summary(model, input_size=(3, 224, 224))

### Method 2 — Truncate VGG19 and rebuild the head

This method removes everything after layer 26, including some convolution layers, the average pooling layer, and the full classifier. A new neural network head is then defined and trained from scratch.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

# ------------------------------------------------------------
# 1. Load pretrained VGG19 and cut after layer 26 (index 26)
# ------------------------------------------------------------
print("Loading pretrained VGG19 and cutting after layer 26...")
full_vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)

# The 'features' module contains all conv+relu+pool layers.
# We keep only layers 0 through 26 (inclusive).
truncated_features = full_vgg.features[:27]  # because Python slice stops before 27

# Optional: freeze the truncated part (transfer learning)
for param in truncated_features.parameters():
    param.requires_grad = False   # set to True if you want to fine-tune

# ------------------------------------------------------------
# 2. Build the custom head (large stride conv, pool, sharp ANN)
# ------------------------------------------------------------
# After truncated_features, the output shape is:
#   channels = 512, height = 28, width = 28
# because the last operation was a ReLU after a conv (no extra pooling).
print("Expected feature map size after truncation: 512 x 28 x 28")

custom_head = nn.Sequential(
    # Large stride convolution to quickly reduce spatial size
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=2, padding=1),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True),
    # Another conv with stride 2 to get even smaller (or use pooling)
    nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True),
    # Adaptive pooling to a fixed small size (e.g., 4x4)
    nn.AdaptiveAvgPool2d((4, 4)),
    # Flatten
    nn.Flatten(),
    # Sharp feed‑forward ANN (few but wide layers)
    nn.Linear(512 * 4 * 4, 2048),   # 512*16 = 8192 -> 2048
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(2048, 1024),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(1024, 100)            # CIFAR-100 output
)

# ------------------------------------------------------------
# 3. Combine truncated VGG features + custom head
# ------------------------------------------------------------
class TruncatedVGG19(nn.Module):
    def __init__(self, features, head):
        super().__init__()
        self.features = features
        self.head = head

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

model = TruncatedVGG19(truncated_features, custom_head)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model architecture:\n", model)
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# ------------------------------------------------------------
# 4. Example of feeding a random batch to verify dimensions
# ------------------------------------------------------------
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 224, 224).to(device)
    output = model(dummy_input)
    print(f"Output shape for batch size 2: {output.shape}")  # should be [2, 100]


# After building the model, check a few layers
for name, param in model.named_parameters():
    if 'features' in name:
        print(f"{name}: requires_grad = {param.requires_grad}")

# Check the first conv layer of the truncated features
print(model.features[0].weight.requires_grad)   # False
# Check a parameter in the head
print(model.head[0].weight.requires_grad)       # True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non‑trainable parameters: {non_trainable_params:,}")

### Method 3 — Fine-tune selected convolution layers

After loading VGG19 and replacing the final classifier layer, freeze all parameters and then unfreeze the first convolutional layer, one middle convolutional layer, and one later convolutional layer for manual fine-tuning.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

# ------------------------------------------------------------
# Step 1: Load full VGG19 with pre-trained weights
# ------------------------------------------------------------
model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)

# ------------------------------------------------------------
# Step 2: Freeze EVERY parameter in the model
# ------------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Step 3: Unfreeze only the specified convolutional layers
# ------------------------------------------------------------
# Layer indices in model.features:
#   0 : Conv2d(3,64)
#  10 : Conv2d(128,256)
#  25 : Conv2d(512,512)
layers_to_unfreeze = [0, 10, 25]

for idx in layers_to_unfreeze:
    # The layer at this index might be a Conv2d (or sometimes ReLU, but here they are Conv2d)
    # We unfreeze its weight and bias parameters
    conv_layer = model.features[idx]
    if isinstance(conv_layer, nn.Conv2d):
        conv_layer.weight.requires_grad = True
        if conv_layer.bias is not None:
            conv_layer.bias.requires_grad = True
        print(f"Unfrozen layer {idx}: {conv_layer}")
    else:
        print(f"Warning: layer {idx} is {conv_layer}, not a Conv2d")

# ------------------------------------------------------------
# Step 4: Redefine the classifier for CIFAR-100 (100 classes)
# ------------------------------------------------------------
num_features = model.classifier[6].in_features   # = 4096
model.classifier[6] = nn.Linear(num_features, 100)

# The new classifier head is trainable by default (requires_grad = True)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ------------------------------------------------------------
# Step 5: Print parameter counts (trainable vs non-trainable)
# ------------------------------------------------------------
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non‑trainable parameters: {non_trainable_params:,}")

# Optionally, verify which layers are trainable
print("\nVerification: Checking requires_grad for selected layers:")
print(f"  Layer 0 (Conv2d 3→64) weight requires_grad: {model.features[0].weight.requires_grad}")
print(f"  Layer 10 (Conv2d 128→256) weight requires_grad: {model.features[10].weight.requires_grad}")
print(f"  Layer 25 (Conv2d 512→512) weight requires_grad: {model.features[25].weight.requires_grad}")
print(f"  Last Linear layer (classifier[6]) weight requires_grad: {model.classifier[6].weight.requires_grad}")

## 6. Train the modified model

In [ ]:
# ------------------------------
# 5. Train the model with CUDA
# ------------------------------
print("\nStep 5: Training the model...")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
# Optional: reduce LR on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

num_epochs = 10  # For demonstration; you can increase for better accuracy

train_losses = []
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if i % 100 == 99:   # print every 100 batches
            print(f"Epoch {epoch+1}, Batch {i+1}: Loss = {loss.item():.4f}")

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100. * correct / total
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs} -> Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

    # Validation after each epoch
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= len(testloader)
    val_acc = 100. * val_correct / val_total
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")
    scheduler.step(val_loss)  # adjust learning rate

print("Training finished.")

## 7. Save the trained model parameters

This example saves the entire model object. It is convenient, but not the safest or most portable method.

In [ ]:
# ---------- Save the entire model (method 2) ----------
torch.save(model, 'truncated_vgg19_full.pth')
print("✅ Entire model saved as 'truncated_vgg19_full.pth'")

## 8. Load the trained model and test five random samples

In [ ]:
# ---------- Simulate kernel restart: load and test on 5 random samples ----------
print("\n--- Simulating kernel restart: loading model from disk ---")
# For PyTorch 2.6+, need weights_only=False to load full model with custom classes
loaded_model = torch.load('truncated_vgg19_full.pth', map_location=device, weights_only=False)
loaded_model.eval()


# Pick 5 random test samples
indices = np.random.choice(len(testset), 5, replace=False)
print("\nPredictions on 5 random test samples:")
for idx in indices:
    image_tensor, true_label = testset[idx]          # already normalized (3,224,224)
    input_batch = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        output = loaded_model(input_batch)
        pred = output.argmax(dim=1).item()

    true_name = testset.classes[true_label]
    pred_name = testset.classes[pred]
    print(f"Sample {idx:5d} | True: {true_name:20s} | Predicted: {pred_name:20s}")

    # Optional: show the image (denormalise first)
    def denormalize(tensor, mean, std):
        img = tensor.clone().cpu()
        for c in range(3):
            img[c] = img[c] * std[c] + mean[c]
        return img.clamp(0, 1)

    img_disp = denormalize(image_tensor, [0.485,0.456,0.406], [0.229,0.224,0.225])
    plt.imshow(img_disp.permute(1,2,0))
    plt.title(f"True: {true_name}\nPred: {pred_name}")
    plt.axis('off')
    plt.show()

## Conclusion

At this point, a CIFAR-100 CNN model based on VGG19 transfer learning has been trained successfully.

The same idea can be applied to many different AI models and deployed rapidly across multiple scenarios. More importantly, the modular structure introduced by transfer learning provides a practical foundation for consistency in AI development and industrial deployment. It balances general-purpose market needs with task-specific optimization in a way that is conceptually similar to fine-tuning large language models.

### Contact

For job opportunities or project collaboration: `yucongcai_business@outlook.com`  
For research-related contact: `yucongcai_research@outlook.com`